# Naive Bayes Classifier

A family of probabilistic classifiers based on Bayes' Theorem with a strong ("naive") independence assumption between features. Despite its simplicity, it works surprisingly well for many real-world problems, especially text classification.

## Bayes' Theorem

$$P(y | X) = \frac{P(X | y) \cdot P(y)}{P(X)}$$

Where:
- $P(y | X)$ = **Posterior**: probability of class $y$ given features $X$
- $P(X | y)$ = **Likelihood**: probability of features $X$ given class $y$
- $P(y)$ = **Prior**: probability of class $y$
- $P(X)$ = **Evidence**: probability of features $X$ (normalizing constant)

### The Naive Assumption

Naive Bayes assumes **conditional independence** between features given the class:

$$P(X_1, X_2, ..., X_n | y) = P(X_1 | y) \cdot P(X_2 | y) \cdot ... \cdot P(X_n | y)$$

This is "naive" because features are rarely independent in real life — yet the algorithm often performs well.

## Types of Naive Bayes

| Variant | Feature Distribution | When to Use |
|---------|---------------------|-------------|
| **GaussianNB** | Continuous, normally distributed | Real-valued features (height, temperature) |
| **MultinomialNB** | Counts / frequencies | Text classification (word counts) |
| **BernoulliNB** | Binary (0/1) features | Presence/absence of words, binary features |

### Laplace Smoothing

Adds a small constant $\alpha$ (typically 1) to all counts to avoid zero probabilities:

$$P(x_i | y) = \frac{N_{yi} + \alpha}{N_y + \alpha \cdot n}$$

Where $n$ is the number of possible values for feature $i$.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris, load_breast_cancer, fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import Binarizer

sns.set_style('whitegrid')
print("Libraries imported.")

In [ ]:
# GaussianNB — Continuous features
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)

print("Gaussian Naive Bayes on Iris:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# GaussianNB on Breast Cancer
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target

X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)

gnb_c = GaussianNB()
gnb_c.fit(X_c_train, y_c_train)
y_c_pred = gnb_c.predict(X_c_test)

print("Gaussian Naive Bayes on Breast Cancer:")
print(f"  Accuracy: {accuracy_score(y_c_test, y_c_pred):.4f}")
print(classification_report(y_c_test, y_c_pred, target_names=cancer.target_names))

In [ ]:
# GaussianNB: see the learned parameters
print("Learned parameters per class (Iris):")
print(f"Number of classes: {gnb.classes_}")
print("\nClass priors:", gnb.class_prior_)
print("\nTheta (mean of each feature per class):")
print(pd.DataFrame(gnb.theta_, columns=iris.feature_names))
print("\nSigma (variance of each feature per class):")
print(pd.DataFrame(gnb.sigma_, columns=iris.feature_names))

## Text Classification with Naive Bayes

Naive Bayes is particularly effective for text classification because:
1. Text data has many features (words) — handles high dimensions well
2. Independence assumption is reasonable for word occurrences
3. Fast to train and predict

### Text Vectorization

**CountVectorizer**: creates a matrix of token counts (bag-of-words)

**TfidfVectorizer**: Term Frequency-Inverse Document Frequency
- TF: how often a word appears in a document
- IDF: how rare the word is across all documents
- TF-IDF = TF × IDF — downweights common words, upweights rare/important ones

In [ ]:
# Text classification example with 20 Newsgroups
# Using a subset: alt.atheism vs soc.religion.christian
categories = ['alt.atheism', 'soc.religion.christian']
newsgroups = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42)

print(f"Samples: {len(newsgroups.data)}")
print(f"Classes: {newsgroups.target_names}")
print(f"\nSample email:\n{newsgroups.data[0][:300]}...")

In [ ]:
# Split text data
X_news_train, X_news_test, y_news_train, y_news_test = train_test_split(
    newsgroups.data, newsgroups.target, test_size=0.2, random_state=42, stratify=newsgroups.target
)

# Convert text to TF-IDF features
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_news_train)
X_test_tfidf = vectorizer.transform(X_news_test)

print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")

In [ ]:
# Multinomial Naive Bayes on TF-IDF features
mnb = MultinomialNB(alpha=1.0)  # alpha = Laplace smoothing
mnb.fit(X_train_tfidf, y_news_train)
y_news_pred = mnb.predict(X_test_tfidf)

print("MultinomialNB — News Classification:")
print(f"  Accuracy: {accuracy_score(y_news_test, y_news_pred):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_news_test, y_news_pred, target_names=newsgroups.target_names))

In [ ]:
# Inspecting the most informative features (words)
feature_names = vectorizer.get_feature_names_out()

# For each class, get log-probability of each word
class_labels = newsgroups.target_names
for i, class_name in enumerate(class_labels):
    top_indices = np.argsort(mnb.feature_log_prob_[i])[-10:]
    top_words = [feature_names[idx] for idx in top_indices]
    print(f"Top words for '{class_name}': {', '.join(reversed(top_words))}")

In [ ]:
# BernoulliNB — Binary features
# Binarizing the Iris dataset
X_iris_bin = Binarizer(threshold=X.median(axis=0)).fit_transform(X)

X_bin_train, X_bin_test, y_bin_train, y_bin_test = train_test_split(
    X_iris_bin, y, test_size=0.2, random_state=42, stratify=y
)

bnb = BernoulliNB(alpha=1.0)
bnb.fit(X_bin_train, y_bin_train)
y_bin_pred = bnb.predict(X_bin_test)

print("BernoulliNB on Binarized Iris:")
print(f"  Accuracy: {accuracy_score(y_bin_test, y_bin_pred):.4f}")
print(f"\nNote: BernoulliNB expects binary features (0 or 1).")

In [ ]:
# Compare Vectorizers for text classification
vectorizers = {
    'CountVectorizer': CountVectorizer(stop_words='english', max_features=5000),
    'TfidfVectorizer': TfidfVectorizer(stop_words='english', max_features=5000)
}

for name, vec in vectorizers.items():
    X_train_vec = vec.fit_transform(X_news_train)
    X_test_vec = vec.transform(X_news_test)
    
    mnb_temp = MultinomialNB(alpha=1.0)
    mnb_temp.fit(X_train_vec, y_news_train)
    acc = accuracy_score(y_news_test, mnb_temp.predict(X_test_vec))
    print(f"  {name:20s} accuracy: {acc:.4f}")

In [ ]:
# Effect of Laplace smoothing (alpha)
alphas = [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
alpha_scores = []

for alpha in alphas:
    mnb_temp = MultinomialNB(alpha=alpha)
    mnb_temp.fit(X_train_tfidf, y_news_train)
    score = accuracy_score(y_news_test, mnb_temp.predict(X_test_tfidf))
    alpha_scores.append(score)
    print(f"  alpha={alpha:6.3f}  accuracy={score:.4f}")

best_alpha = alphas[np.argmax(alpha_scores)]
print(f"\nBest alpha: {best_alpha}")

In [ ]:
# Problem: Spam Email Classification Simulation
# Create a simple spam/ham dataset
spam_emails = [
    "win free money now click here",
    "congratulations you won a prize",
    "limited time offer buy now",
    "click here to claim your reward",
    "act now supplies are limited",
    "meeting tomorrow at 3pm",
    "project update attached document",
    "lunch at noon see you there",
    "can you review the proposal",
    "team standup at 10am"
]
spam_labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

# Vectorize
vec_spam = CountVectorizer()
X_spam = vec_spam.fit_transform(spam_emails)

# Train
mnb_spam = MultinomialNB(alpha=1.0)
mnb_spam.fit(X_spam, spam_labels)

# Test new emails
test_emails = [
    "win cash now",
    "meeting agenda attached",
    "free money clicking here"
]
X_test_spam = vec_spam.transform(test_emails)
predictions = mnb_spam.predict(X_test_spam)
probs = mnb_spam.predict_proba(X_test_spam)

for email, pred, prob in zip(test_emails, predictions, probs):
    label = "SPAM" if pred == 1 else "HAM"
    print(f"  '{email:30s}' -> {label} (spam prob: {prob[1]:.3f})")

In [ ]:
# Pros and Cons of Naive Bayes
print("""
PROS:
- Extremely fast to train and predict
- Works well with high-dimensional data (text, images)
- Performs well even with limited training data
- Handles both categorical and continuous features
- Naturally handles multi-class classification
- Less prone to overfitting

CONS:
- Naive independence assumption is often violated
- If a feature category is not seen in training, Laplace smoothing needed
- Can't learn feature interactions
- Often outperformed by more complex models (RF, GBM) for tabular data
- Probabilistic outputs are not well-calibrated
""")

In [ ]:
# Interview Problem: Why does Naive Bayes work well for text?
print("""
Interview Question: Why does Naive Bayes work well for text classification
despite the independence assumption being obviously false?

Answer:
1. In text, the feature space is very high-dimensional (word counts).
   Naive Bayes handles this efficiently.
2. The independence assumption, while false, doesn't hurt much because
   what matters is the RANKING of probabilities, not the raw values.
3. Even if P(X|y) is poorly estimated, the relative comparisons between
   classes may still be correct.
4. With enough training data, the model captures the distinguishing words
   for each category effectively.
5. It serves as a strong baseline that can be deployed quickly.
""")

In [ ]:
# Summary
print("=" * 60)
print("NAIVE BAYES — KEY TAKEAWAYS")
print("=" * 60)
print("""
1. Based on Bayes' Theorem with conditional independence assumption
2. GaussianNB: continuous features (normal distribution)
3. MultinomialNB: count/frequency features (text classification)
4. BernoulliNB: binary features (word presence/absence)
5. Laplace smoothing (alpha) prevents zero probability issues
6. Excellent for text classification with CountVectorizer/TfidfVectorizer
7. Very fast, handles high dimensions, good with small data
""")